# Jailbreak Explanation Experiment

Runs 15 (prompt/response pairs) x 3 (scoring modes) = 45 `JailbreakGame` runs and computes first-order Shapley values via KernelSHAP.


In [5]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import shapiq

from JailbreakAnalysisGame import JailbreakGame
from demos.shared.hf_model import HFModelWrapper

## Config

In [ ]:
SUMMARY_PATH = Path("results/summary_asr.json")
OUTPUT_PATH = Path("results/explanation_experiment_data.json")

TARGET_MODEL =  "TinyLlama/TinyLlama-1.1B-Chat-v1.0" #"google/gemma-4-e4b-it"
TARGET_TEMPERATURE = 0.7
JUDGE_MODEL = "openai/gpt-oss-safeguard-20b"
N_ENTRIES = 15
DEVICE = "cuda"

# Running locally for now: llm-as-a-judge needs a second, large judge model loaded
# alongside the main model, which isn't feasible on this machine. Flip this back on
# once you can run on something with enough memory for both models simultaneously.
RUN_JUDGE_MODE = False
SCORING_MODES = ["abs-logprob", "contra-logprob"] + (
    ["llm-as-a-judge"] if RUN_JUDGE_MODE else []
)

## Budget helper (copied from your app code)

In [3]:
def recommended_budget(n_players: int, *, second_order: bool, multiplier: float = 1.0) -> int:
    """Pick a coalition budget that scales with players and interaction order."""
    n_coeff = n_players + n_players * (n_players - 1) // 2 if second_order else n_players
    budget = int((4 * n_coeff + 2) * multiplier)
    budget = max(budget, n_coeff + 2)
    if n_players <= 20:
        budget = min(budget, 2**n_players)
    return budget

## Load the 15 target entries

`summary_asr.json` is a flat list of records, filtered by `model` and `temperature`.

In [4]:
def load_target_entries(path: Path, n: int = N_ENTRIES) -> list[dict]:
    with path.open() as f:
        entries: list[dict] = json.load(f)  # flat list of records

    filtered = [
        e
        for e in entries
        if e.get("model") == TARGET_MODEL and e.get("temperature") == TARGET_TEMPERATURE
    ]

    if len(filtered) < n:
        msg = (
            f"Expected at least {n} entries for model={TARGET_MODEL!r}, "
            f"temperature={TARGET_TEMPERATURE}, found {len(filtered)}."
        )
        raise ValueError(msg)

    return filtered[:n]

In [7]:
entries = load_target_entries(SUMMARY_PATH)
len(entries)

15

## Run a single game -> full-coalition score, baseline, first-order Shapley values, and second-order (pairwise) k-SII interactions

In [8]:
def run_single_game(
    prompt_text: str,
    response: str,
    scoring_mode: str,
    hf_model: HFModelWrapper,
) -> dict:
    game = JailbreakGame(
        model_name=TARGET_MODEL,
        input_text=prompt_text,
        scoring_mode=scoring_mode,
        judge_model_name=JUDGE_MODEL,
        model_response=response,
        device=DEVICE,
        hf_model=hf_model,  # reuse already-loaded model instead of reloading per game
    )

    # -------------------------
    # 1. Final (full-coalition) value function score
    # -------------------------
    full_coalition = np.ones((1, game.n_players), dtype=bool)
    full_value = float(game.value_function(full_coalition)[0])

    # -------------------------
    # 2 & 3. First-order Shapley values (+ baseline, as a byproduct)
    # -------------------------
    budget_order1 = recommended_budget(game.n_players, second_order=False)
    approx_order1 = shapiq.KernelSHAP(n=game.n_players, random_state=42)
    result_order1 = approx_order1.approximate(budget=budget_order1, game=game)

    baseline_value = float(result_order1.baseline_value)
    shapley_values = np.asarray(result_order1.values).tolist()

    # -------------------------
    # 4. Second-order (pairwise) k-SII interaction values
    # -------------------------
    budget_order2 = recommended_budget(game.n_players, second_order=True)
    approx_order2 = shapiq.KernelSHAPIQ(
        n=game.n_players,
        index="k-SII",
        max_order=2,
        random_state=42,
    )
    result_order2 = approx_order2.approximate(budget=budget_order2, game=game)

    order2_lookup = {k: v for k, v in result_order2.interaction_lookup.items() if len(k) == 2}
    interaction_values = [
        {
            "players": [str(game.players[i]) for i in player_idx],
            "value": float(result_order2.values[pos]),
        }
        for player_idx, pos in order2_lookup.items()
    ]

    return {
        "n_players": game.n_players,
        "players": [str(p) for p in game.players],
        "full_coalition_value": full_value,
        "baseline_value": baseline_value,
        "budget_order1": budget_order1,
        "shapley_values": shapley_values,
        "budget_order2": budget_order2,
        "interaction_values": interaction_values,
    }

## Main experiment loop: 15 entries x 3 scoring modes = 45 runs

In [9]:
def run_experiment(entries: list[dict]) -> list[dict]:
    results = []
    total = len(entries) * len(SCORING_MODES)
    step = 0

    # Load the main model once and reuse it across all 15*len(SCORING_MODES) games,
    # instead of reloading it fresh inside every JailbreakGame(...) construction.
    print(f"Loading {TARGET_MODEL} once for reuse across all games...")
    shared_model = HFModelWrapper(TARGET_MODEL, device=DEVICE)

    for i, entry in enumerate(entries):
        prompt_text = entry["prompt_text"]
        response = entry["response"]

        for scoring_mode in SCORING_MODES:
            step += 1
            print(f"[{step}/{total}] entry={i} scoring_mode={scoring_mode}")

            game_result = run_single_game(prompt_text, response, scoring_mode, shared_model)

            results.append(
                {
                    "entry_index": i,
                    "model": TARGET_MODEL,
                    "temperature": TARGET_TEMPERATURE,
                    "judge_model": JUDGE_MODEL,
                    "prompt_text": prompt_text,
                    "response": response,
                    "scoring_mode": scoring_mode,
                    **game_result,
                }
            )

    return results

In [ ]:
results = run_experiment(entries)
len(results)

Loading google/gemma-4-e4b-it once for reuse across all games...
[CausalModelWrapper] Loading 'google/gemma-4-e4b-it' on mps


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


: 

## Save results

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} game results ({len(entries)}x{len(SCORING_MODES)}) to {OUTPUT_PATH}")

## Quick sanity check

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df[["entry_index", "scoring_mode", "n_players", "budget"]]